# ViT master experiments: generator diversity and augmentation

**Based on `ViT_Team_k2_Starter.ipynb`.** Frozen OpenAI CLIP ViT-L/14 encodes images;
a new linear real/fake classifier is trained for every experiment. This is supervised
classification, not zero-shot CLIP or full ViT fine-tuning.

**Default plan:** ADM held out, k = 1/2/4/7, repeats = 0/1/2, seed = 17,
with `baseline` and one `augmented` recipe: **24 training runs**. Nothing is assumed
completed. Existing starter-notebook runs are not imported or overwritten.

1. Select the **CSCI566 ViT** kernel in a CARC GPU-backed OnDemand session.
2. Configure everything in Section 1. Keep the dataset team's CSVs unchanged.
3. Run all cells in order. Section 8 runs the plan; Section 9 evaluates saved best heads.
4. Re-run with the same settings to resume. Finished runs are verified and skipped;
interrupted training resumes at the next unfinished epoch. Use a new study name for
intentional changes to the scientific protocol.

Training and checkpoint selection use only train/validation. Final test evaluation
does not change model selection, thresholds, or calibration. Agreement with the CNN
team on these CSVs and augmentation settings is required for matched comparisons.

## 1. All experiment settings

Change settings here, then rerun all cells below. Paths are CARC defaults from the README.
`REPEATS` selects dataset manifests; `SEEDS` controls classifier initialization/training.
They are different sources of variation. `HOLDOUTS` can include other generators **after**
the dataset team provides their manifests. The default recipe is a combined augmentation
condition, not separate blur/JPEG/noise ablations.

In [ ]:
from pathlib import Path

# Storage: dataset is read-only; outputs and caches are written under project storage.
SHARED_ROOT = Path('/project2/yzhao010_1531/csci566-genimage')
DATA_ROOT = SHARED_ROOT / 'data/extracted/GenImage_Compact'
MANIFEST_ROOT = DATA_ROOT / '_experiment_manifests'
STUDY_NAME = 'vit_master_v1'
OUTPUT_ROOT = SHARED_ROOT / 'results' / STUDY_NAME
FEATURE_CACHE_ROOT = SHARED_ROOT / 'features' / STUDY_NAME

# Plan: 1 holdout x 4 k values x 3 dataset repeats x 1 seed x 2 modes = 24 runs.
HOLDOUTS = ['ADM']
K_VALUES = [1, 2, 4, 7]
REPEATS = [0, 1, 2]
SEEDS = [17]
TRAINING_MODES = ['baseline', 'augmented']
# Optionally restrict by exact folder name; [] means all combinations above.
EXPERIMENT_ALLOWLIST = []

# Model and classifier: same baseline defaults as the existing ViT starter.
MODEL_NAME = 'ViT-L/14'
CLIP_DOWNLOAD_ROOT = Path.home() / '.cache/clip'
IMAGE_BATCH_SIZE = 16       # Use 8 or 4 if GPU memory runs out.
HEAD_BATCH_SIZE = 128
MAX_EPOCHS = 20
PATIENCE = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
THRESHOLD = 0.5
NUM_WORKERS = 0             # Notebook-safe default.
REQUIRE_GPU = True

# Runtime and restart behavior. Change STUDY_NAME to intentionally start a new study.
RUN_TRAINING = True
RUN_FINAL_TEST = True       # Set False to train/validate without final test reporting.
RESUME = True
CONTINUE_ON_ERROR = False   # Fail immediately by default; errors are logged either way.
VERIFY_IMAGE_HASHES = True  # Verify manifest SHA-256 against actual bytes once per invocation.
USE_FEATURE_CACHE = True    # Deterministic features only; stochastic train views are never cached.
MAX_RUNS_THIS_SESSION = None  # e.g. 2; None runs the whole selected plan.

# Baseline has only CLIP's normal resize/center-crop/normalization.
# Augmented mode: on each training image, each operation is independently applied
# with its own probability, in the order below. Fresh views are drawn every epoch.
# Probabilities apply to BOTH real and fake images. Validation/test stay deterministic.
AUGMENTATION = {
    'blur_p': 0.3, 'blur_radius': [0.1, 2.0],
    'jpeg_p': 0.3, 'jpeg_quality': [40, 95],
    'resize_p': 0.3, 'resize_scale': [0.5, 1.0],
    'noise_p': 0.3, 'noise_sigma': [0.005, 0.03],  # pixels scaled to [0, 1]
    'lowpass_p': 0.3, 'lowpass_cutoff': [0.35, 0.9],  # fraction of axis Nyquist frequency
}

# Optional robustness evaluation adds evaluations, not training runs.
# Keep ['clean'] for the initial 24-run benchmark. Add named conditions later to
# evaluate existing checkpoints without retraining. Conditions are applied to both labels.
EVAL_CONDITIONS = ['clean']  # available: clean, blur, jpeg, resize, noise, lowpass
EVAL_DEGRADATIONS = {
    'blur_radius': 1.0, 'jpeg_quality': 60, 'resize_scale': 0.5,
    'noise_sigma': 0.02, 'lowpass_cutoff': 0.6,
}
EVALUATION_SEED = 566
CALIBRATION_BINS = 10       # Reliability diagram: mean prob_fake vs observed fake fraction.
# Report raw probability calibration: ECE, Brier, log loss. No fitted calibration.

# Expected counts per class in the dataset team's experiment CSVs.
EXPECTED_COUNTS = {'train': {0: 1400, 1: 1400},
                   'val': {0: 280, 1: 280}, 'test': {0: 280, 1: 280}}
GENERATOR_FAMILIES = {
    'ADM': 'diffusion', 'BigGAN': 'GAN', 'glide': 'diffusion',
    'stable_diffusion_v_1_4': 'diffusion', 'stable_diffusion_v_1_5': 'diffusion',
    'VQDM': 'diffusion', 'wukong': 'diffusion', 'Midjourney': 'unassigned',
}
# Midjourney's exact model/version is not specified in these manifests; do not
# silently assign it a family. Family summaries need multiple corresponding holdouts.


## 2. Environment and GPU check

Use the existing `setup_carc_vit.sh` once if packages are missing. This notebook never
installs packages into a running kernel. A real GPU operation is checked below.

In [ ]:
import contextlib
import copy
import hashlib
import importlib.metadata
import inspect
import io
import itertools
import json
import math
import os
import platform
import random
import re
import subprocess
import sys
import time
import traceback
import uuid
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
from IPython.display import display
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
from tqdm import tqdm
import clip

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python:', sys.executable)
print('PyTorch:', torch.__version__, '| device:', DEVICE)
if REQUIRE_GPU and DEVICE.type != 'cuda':
    raise RuntimeError('Select CSCI566 ViT in a GPU-backed OnDemand session before running.')
if DEVICE.type == 'cuda':
    x = torch.ones((16, 16), device=DEVICE)
    assert (x @ x).sum().item() == 4096
    torch.cuda.synchronize()
    print('GPU operation passed:', torch.cuda.get_device_name(0))


## 3. Utilities, protocol validation, and run identities

Each experiment/mode/seed has one stable run directory. Matching completed runs are
skipped; configuration or code changes are rejected instead of overwriting results.
Only one process may write a run at a time. If a CARC job is killed, Section 8 explains
how to remove its stale lock after confirming the old job is no longer running.

In [ ]:
ENGINE_VERSION = '1.0.0'

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(type(value).__name__)

def digest_json(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=json_default).encode()).hexdigest()

def atomic_json(path, value):
    path = Path(path)
    temp = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    try:
        temp.write_text(json.dumps(value, indent=2, default=json_default, allow_nan=False))
        os.replace(temp, path)
    finally:
        temp.unlink(missing_ok=True)

def atomic_csv(path, frame):
    path = Path(path)
    temp = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    try:
        frame.to_csv(temp, index=False)
        os.replace(temp, path)
    finally:
        temp.unlink(missing_ok=True)

def atomic_torch(path, value):
    path = Path(path)
    temp = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    try:
        torch.save(value, temp)
        os.replace(temp, path)
    finally:
        temp.unlink(missing_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

@contextlib.contextmanager
def run_lock(directory):
    path = directory / 'RUNNING.lock'
    try:
        fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o664)
    except FileExistsError:
        raise RuntimeError(f'Run is locked: {path}. Confirm its previous job has stopped before removing the lock.')
    try:
        with os.fdopen(fd, 'w') as file:
            json.dump({'pid': os.getpid(), 'host': platform.node(), 'started_utc': utc_now()}, file)
        yield
    finally:
        path.unlink(missing_ok=True)

def validate_settings():
    for name, values in [('HOLDOUTS', HOLDOUTS), ('K_VALUES', K_VALUES),
                         ('REPEATS', REPEATS), ('SEEDS', SEEDS), ('TRAINING_MODES', TRAINING_MODES),
                         ('EVAL_CONDITIONS', EVAL_CONDITIONS)]:
        if not values or len(values) != len(set(values)):
            raise ValueError(f'{name} must be nonempty and contain no duplicates.')
    if not set(K_VALUES) <= {1, 2, 4, 7} or not set(REPEATS) <= {0, 1, 2}:
        raise ValueError('Expected k in 1,2,4,7 and dataset repeat in 0,1,2.')
    if not set(TRAINING_MODES) <= {'baseline', 'augmented'}:
        raise ValueError('TRAINING_MODES must use baseline/augmented.')
    if not set(EVAL_CONDITIONS) <= {'clean', 'blur', 'jpeg', 'resize', 'noise', 'lowpass'}:
        raise ValueError('Unknown evaluation condition.')
    if not (0 < THRESHOLD < 1) or not (1 <= CALIBRATION_BINS <= 100):
        raise ValueError('Invalid threshold or calibration bin count.')
    if any(v <= 0 for v in [IMAGE_BATCH_SIZE, HEAD_BATCH_SIZE, MAX_EPOCHS, PATIENCE, LEARNING_RATE]):
        raise ValueError('Training sizes, patience and learning rate must be positive.')
    if NUM_WORKERS != 0:
        raise ValueError('Use NUM_WORKERS=0 for reproducible per-epoch augmented views in this notebook.')
    if MAX_RUNS_THIS_SESSION is not None and MAX_RUNS_THIS_SESSION < 1:
        raise ValueError('MAX_RUNS_THIS_SESSION must be positive or None.')
    for key, value in AUGMENTATION.items():
        if key.endswith('_p') and not 0 <= value <= 1:
            raise ValueError(f'{key} must be in [0,1].')
        if isinstance(value, list) and (len(value) != 2 or value[0] > value[1]):
            raise ValueError(f'Invalid range: {key}')
    for config in [AUGMENTATION, EVAL_DEGRADATIONS]:
        for key, value in config.items():
            vals = value if isinstance(value, list) else [value]
            if key.endswith('_p'):
                continue
            if any(v < 0 for v in vals):
                raise ValueError(f'{key} cannot be negative.')
            if ('scale' in key or 'cutoff' in key) and any(not 0 < v <= 1 for v in vals):
                raise ValueError(f'{key} must be in (0,1].')
            if 'quality' in key and any(not 1 <= v <= 100 for v in vals):
                raise ValueError('JPEG quality must be in [1,100].')
    if not DATA_ROOT.is_dir() or not MANIFEST_ROOT.is_dir():
        raise FileNotFoundError(f'Dataset/manifests not found under {DATA_ROOT}.')
    data_resolved = DATA_ROOT.resolve()
    for path in [OUTPUT_ROOT, FEATURE_CACHE_ROOT]:
        if path.resolve().is_relative_to(data_resolved):
            raise ValueError('Outputs/caches must be outside the read-only dataset.')
        path.mkdir(parents=True, exist_ok=True)

def package_versions():
    versions = {'python': platform.python_version(), 'torch': str(torch.__version__)}
    for package in ['torchvision', 'numpy', 'pandas', 'scikit-learn', 'pillow', 'clip']:
        versions[package] = importlib.metadata.version(package)
    versions['clip_source'] = json.loads(
        importlib.metadata.distribution('clip').read_text('direct_url.json') or '{}')
    return versions

def code_fingerprint():
    # inspect reads notebook cell definitions; it also works in the standalone QA script.
    names = ['seed_everything', 'resolve_image_path', 'load_experiment', 'file_digest', 'lowpass_image', 'corrupt_image',
             'ImageTransform', 'ManifestImages', 'feature_identity', 'deterministic_features',
             'encode_batch', 'image_loader', 'export_predictions', 'evaluation_identity',
             'reliability_bins', 'classification_metrics', 'train_one', 'predict_features',
             'evaluate_one']
    return digest_json({'version': ENGINE_VERSION,
                        'functions': {name: inspect.getsource(globals()[name]) for name in names}})


## 4. Read and verify the dataset team's CSVs

Verify class counts, generator identities, path separation, SHA-256 separation, and
image availability before training. By default, verify actual file hashes as well.
Paths from Colab are remapped to the configured CARC dataset root. No new splits are made.
Exact hashes do not detect near-duplicate images; the dataset team owns that audit.

In [ ]:
VERIFIED_CONTENT = {}

def resolve_image_path(raw_path):
    text = str(raw_path).replace('\\', '/')
    marker = re.search(r'(?:^|/)GenImage_Compact(?:_shared)?/', text)
    if marker:
        path = DATA_ROOT / text[marker.end():]
    else:
        path = Path(text)
        path = path if path.is_absolute() else DATA_ROOT / path
    path = path.resolve()
    if not path.is_relative_to(DATA_ROOT.resolve()):
        raise ValueError(f'Manifest path is outside DATA_ROOT: {raw_path}')
    return path

def file_digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as file:
        for block in iter(lambda: file.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def load_experiment(experiment):
    match = re.fullmatch(r'holdout_(.+)_k(1|2|4|7)_rep([012])', experiment)
    if not match:
        raise ValueError(f'Unexpected experiment name: {experiment}')
    holdout, k, rep = match.group(1), int(match.group(2)), int(match.group(3))
    frames, checksums = {}, {}
    for split in ['train', 'val', 'test']:
        csv = MANIFEST_ROOT / experiment / f'{split}.csv'
        checksums[split] = file_digest(csv)
        frame = pd.read_csv(csv, dtype={'sha256': str})
        required = {'path', 'label', 'source_generator', 'sha256'}
        if not required <= set(frame):
            raise ValueError(f'{csv}: missing {required - set(frame)}')
        if frame[list(required)].isna().any().any() or not frame.label.isin([0, 1]).all():
            raise ValueError(f'{csv}: missing values or invalid labels.')
        frame['label'] = frame.label.astype(int)
        frame['sha256'] = frame.sha256.str.lower()
        if not frame.sha256.str.fullmatch(r'[0-9a-f]{64}').all():
            raise ValueError(f'{csv}: invalid content hash.')
        if frame.label.value_counts().to_dict() != EXPECTED_COUNTS[split]:
            raise ValueError(f'{csv}: unexpected counts {frame.label.value_counts().to_dict()}')
        frame['resolved_path'] = frame.path.map(lambda p: str(resolve_image_path(p)))
        for row in frame.itertuples():
            path = Path(row.resolved_path)
            if not path.is_file():
                raise FileNotFoundError(path)
            if VERIFY_IMAGE_HASHES:
                stat = path.stat()
                key = (str(path), stat.st_size, stat.st_mtime_ns)
                if key not in VERIFIED_CONTENT:
                    VERIFIED_CONTENT[key] = file_digest(path)
                if VERIFIED_CONTENT[key] != row.sha256:
                    raise ValueError(f'Content differs from manifest: {path}')
        frames[split] = frame
    for column in ['resolved_path', 'sha256']:
        if not pd.concat([f[column] for f in frames.values()]).is_unique:
            raise ValueError(f'{experiment}: duplicate {column} within or across splits.')
    generators = {s: set(f.loc[f.label == 1, 'source_generator']) for s, f in frames.items()}
    if len(generators['train']) != k or generators['train'] != generators['val']:
        raise ValueError(f'{experiment}: training/validation generators do not match k.')
    if generators['test'] != {holdout} or holdout in generators['train']:
        raise ValueError(f'{experiment}: held-out generator leaked into training/validation.')
    nature_gens = set(pd.concat([f.loc[f.label == 0, 'source_generator'] for f in frames.values()]))
    if len(nature_gens) != 1:
        raise ValueError(f'{experiment}: expected one canonical real-image source, found {nature_gens}')
    return frames, {'experiment': experiment, 'holdout': holdout, 'k': k, 'rep': rep,
                    'train_generators': sorted(generators['train']),
                    'nature_generator': next(iter(nature_gens)), 'manifest_sha256': checksums}


## 5. Frozen CLIP, augmentation, and deterministic feature caching

The baseline preserves the starter's CLIP preprocessing and L2-normalized features.
Augmentation happens after CLIP's resize/center-crop and before tensor conversion and
normalization. The encoder remains frozen and in evaluation mode in both conditions.
Augmented training encodes new image views every epoch. Only deterministic baseline,
validation, and evaluation features are cached. Caches are keyed by ordered image
hashes, model/preprocessing, implementation, package versions, and degradation settings.

In [ ]:
def lowpass_image(image, cutoff):
    array = np.asarray(image, dtype=np.float32) / 255.0
    height, width = array.shape[:2]
    fy = np.fft.fftfreq(height)[:, None]
    fx = np.fft.fftfreq(width)[None, :]
    mask = (np.sqrt(fx * fx + fy * fy) <= 0.5 * cutoff)[..., None]
    filtered = np.fft.ifft2(np.fft.fft2(array, axes=(0, 1)) * mask, axes=(0, 1)).real
    return Image.fromarray(np.round(np.clip(filtered, 0, 1) * 255).astype(np.uint8))

def corrupt_image(image, operation, value, rng):
    if operation == 'blur':
        return image.filter(ImageFilter.GaussianBlur(radius=value))
    if operation == 'jpeg':
        with io.BytesIO() as buffer:
            image.save(buffer, format='JPEG', quality=int(value), subsampling=2)
            buffer.seek(0)
            with Image.open(buffer) as decoded:
                return decoded.convert('RGB').copy()
    if operation == 'resize':
        width, height = image.size
        return image.resize((max(1, round(width * value)), max(1, round(height * value))),
                            Image.Resampling.BICUBIC).resize(image.size, Image.Resampling.BICUBIC)
    if operation == 'noise':
        array = np.asarray(image, dtype=np.float32) / 255.0
        noisy = np.clip(array + rng.normal(0, value, array.shape), 0, 1)
        return Image.fromarray(np.round(noisy * 255).astype(np.uint8))
    if operation == 'lowpass':
        return lowpass_image(image, value)
    raise ValueError(operation)

class ImageTransform:
    def __init__(self, preprocess, augmented=False, condition='clean'):
        # Official OpenAI CLIP transform: Resize, CenterCrop, RGB, ToTensor, Normalize.
        parts = preprocess.transforms
        self.geometry = transforms.Compose(parts[:2])
        self.finish = transforms.Compose(parts[2:])
        self.augmented, self.condition = augmented, condition

    def __call__(self, image, image_hash):
        image = self.geometry(image)
        if self.augmented:
            specs = [('blur', 'blur_radius'), ('jpeg', 'jpeg_quality'), ('resize', 'resize_scale'),
                     ('noise', 'noise_sigma'), ('lowpass', 'lowpass_cutoff')]
            for operation, key in specs:
                if random.random() < AUGMENTATION[operation + '_p']:
                    lo, hi = AUGMENTATION[key]
                    value = random.randint(int(lo), int(hi)) if operation == 'jpeg' else random.uniform(lo, hi)
                    image = corrupt_image(image, operation, value, np.random)
        elif self.condition != 'clean':
            # Content-based seed makes degradation independent of batch order/model/run.
            seed = int(hashlib.sha256(f'{EVALUATION_SEED}:{image_hash}:{self.condition}'.encode()).hexdigest()[:16], 16)
            rng = np.random.default_rng(seed)
            keys = {'blur': 'blur_radius', 'jpeg': 'jpeg_quality', 'resize': 'resize_scale',
                    'noise': 'noise_sigma', 'lowpass': 'lowpass_cutoff'}
            image = corrupt_image(image, self.condition, EVAL_DEGRADATIONS[keys[self.condition]], rng)
        return self.finish(image)

class ManifestImages(Dataset):
    def __init__(self, frame, transform):
        self.frame, self.transform = frame.reset_index(drop=True), transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.resolved_path) as image:
            image = self.transform(image.convert('RGB'), row.sha256)
        return image, float(row.label)

ENCODER = PREPROCESS = None
ENVIRONMENT = None
ENGINE_HASH = None

def ensure_encoder():
    global ENCODER, PREPROCESS
    if ENCODER is None:
        ENCODER, PREPROCESS = clip.load(MODEL_NAME, device=DEVICE, jit=False,
                                        download_root=str(CLIP_DOWNLOAD_ROOT))
        ENCODER.eval()
        for parameter in ENCODER.parameters():
            parameter.requires_grad_(False)
    return ENCODER, PREPROCESS

def feature_identity(frame, condition):
    return {'images': frame.sha256.tolist(), 'model': MODEL_NAME,
            'preprocessing': repr(PREPROCESS), 'l2_normalized': True,
            'condition': condition,
            'degradation': EVAL_DEGRADATIONS if condition != 'clean' else None,
            'evaluation_seed': EVALUATION_SEED if condition != 'clean' else None,
            'engine': ENGINE_HASH, 'environment': ENVIRONMENT,
            'device_type': DEVICE.type, 'image_batch_size': IMAGE_BATCH_SIZE}

@torch.no_grad()
def encode_batch(images):
    features = F.normalize(ENCODER.encode_image(images.to(DEVICE)).float(), dim=-1)
    if not torch.isfinite(features).all():
        raise RuntimeError('Nonfinite CLIP features.')
    return features

def image_loader(frame, transform, shuffle=False, seed=0):
    return DataLoader(ManifestImages(frame, transform), batch_size=IMAGE_BATCH_SIZE,
                      shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda',
                      generator=torch.Generator().manual_seed(seed))

@torch.no_grad()
def deterministic_features(frame, condition='clean'):
    ensure_encoder()
    key = digest_json(feature_identity(frame, condition))
    path = FEATURE_CACHE_ROOT / f'{key}.pt'
    if USE_FEATURE_CACHE and path.exists():
        cached = torch.load(path, map_location='cpu', weights_only=True)
        expected_labels = torch.tensor(frame.label.to_numpy(), dtype=torch.float32)
        if (cached['key'] != key or cached['features'].shape[0] != len(frame)
                or not torch.equal(cached['labels'], expected_labels)
                or not torch.isfinite(cached['features']).all()):
            raise ValueError(f'Invalid feature cache: {path}')
        return cached['features'], cached['labels']
    loader = image_loader(frame, ImageTransform(PREPROCESS, condition=condition))
    xs, ys = [], []
    for images, labels in tqdm(loader, desc=f'CLIP {condition}', leave=False):
        xs.append(encode_batch(images).cpu())
        ys.append(labels.float())
    features, labels = torch.cat(xs), torch.cat(ys)
    if USE_FEATURE_CACHE:
        atomic_torch(path, {'key': key, 'features': features, 'labels': labels})
    return features, labels


## 6. Common evaluation metrics and calibration

Real = 0; fake = 1. F1/precision/recall refer to fake. AUROC uses probabilities.
Calibration uses equal-width bins of **fake probability**, including p=0 and p=1;
ECE weights each bin's absolute probability/fake-frequency gap by its sample count.
Brier score and binary log loss are also reported. No thresholds or temperatures are
fitted on test data. Generator/family tables pair fake images with the same real pool;
metrics such as AUROC/F1 are never computed on fake-only groups.

In [ ]:
def reliability_bins(labels, probabilities, bins=10):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    index = np.minimum((probabilities * bins).astype(int), bins - 1)
    rows = []
    for b in range(bins):
        selected = index == b
        count = int(selected.sum())
        rows.append({'bin': b, 'lower': b / bins, 'upper': (b + 1) / bins, 'count': count,
                     'mean_prob_fake': float(probabilities[selected].mean()) if count else None,
                     'observed_fake_fraction': float(labels[selected].mean()) if count else None})
    return pd.DataFrame(rows)

def classification_metrics(labels, probabilities):
    labels, probabilities = np.asarray(labels), np.asarray(probabilities, dtype=float)
    if (labels.ndim != 1 or probabilities.shape != labels.shape or len(labels) == 0
            or not np.isin(labels, [0, 1]).all() or not np.isfinite(probabilities).all()
            or ((probabilities < 0) | (probabilities > 1)).any()):
        raise ValueError('Expected finite binary labels and aligned probabilities in [0,1].')
    labels = labels.astype(int)
    predicted = (probabilities >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    table = reliability_bins(labels, probabilities, CALIBRATION_BINS)
    occupied = table[table['count'] > 0]
    ece = ((occupied['mean_prob_fake'] - occupied['observed_fake_fraction']).abs()
           * occupied['count']).sum() / len(labels)
    clipped = np.clip(probabilities, 1e-7, 1 - 1e-7)
    return {'n': len(labels), 'threshold': THRESHOLD,
            'accuracy': float(accuracy_score(labels, predicted)),
            'precision_fake': float(precision_score(labels, predicted, zero_division=0)),
            'recall_fake': float(recall_score(labels, predicted, zero_division=0)),
            'f1_fake': float(f1_score(labels, predicted, zero_division=0)),
            'auroc': float(roc_auc_score(labels, probabilities)) if len(np.unique(labels)) == 2 else None,
            'ece': float(ece), 'brier': float(np.mean((probabilities - labels) ** 2)),
            'log_loss': float(-np.mean(labels * np.log(clipped) + (1-labels) * np.log1p(-clipped))),
            'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}

@torch.no_grad()
def predict_features(head, features):
    head.eval()
    return torch.cat([torch.sigmoid(head(batch.to(DEVICE)).squeeze(-1)).cpu()
                      for batch in features.split(HEAD_BATCH_SIZE)]).numpy()

def export_predictions(directory, stem, frame, probabilities, metadata):
    result = frame.drop(columns=['resolved_path']).copy()
    result['image_id'] = result.sha256
    result['prob_fake'] = probabilities
    result['pred_label'] = (probabilities >= THRESHOLD).astype(int)
    for key, value in metadata.items():
        result[key] = value
    result['threshold'] = THRESHOLD
    atomic_csv(directory / f'{stem}_predictions.csv', result)
    metrics = classification_metrics(result.label, probabilities)
    atomic_json(directory / f'{stem}_metrics.json', metrics)
    table = reliability_bins(result.label, probabilities, CALIBRATION_BINS)
    atomic_csv(directory / f'{stem}_calibration.csv', table)
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.7))
    occupied = table[table['count'] > 0]
    axes[0].plot([0, 1], [0, 1], '--', color='gray')
    axes[0].plot(occupied.mean_prob_fake, occupied.observed_fake_fraction, 'o-')
    axes[0].set(xlabel='Mean predicted fake probability', ylabel='Observed fake fraction',
                xlim=(0, 1), ylim=(0, 1), title=f'{stem}: ECE={metrics["ece"]:.3f}')
    matrix = np.array([[metrics['tn'], metrics['fp']], [metrics['fn'], metrics['tp']]])
    axes[1].imshow(matrix, cmap='Blues')
    for (i, j), count in np.ndenumerate(matrix):
        axes[1].text(j, i, str(count), ha='center', va='center',
                     color='white' if count > matrix.max()/2 else 'black')
    axes[1].set(xticks=[0, 1], yticks=[0, 1], xticklabels=['Real', 'Fake'],
                yticklabels=['Real', 'Fake'], xlabel='Predicted label', ylabel='True label',
                title=f'Accuracy={metrics["accuracy"]:.3f}, AUROC={metrics["auroc"]:.3f}')
    fig.tight_layout()
    fig.savefig(directory / f'{stem}_diagnostics.png', dpi=150)
    plt.close(fig)
    return metrics


## 7. Train one fresh head, checkpoint each epoch, and evaluate

Baseline training matches the starter's AdamW/BCE setup. Augmented image features are
generated every epoch, then the head uses the same head minibatch size as baseline.
The backbone never receives gradients. A completed epoch is an atomic resume point:
if interrupted mid-epoch, that epoch is repeated with the same deterministic seed.
The best epoch uses validation AUROC, then validation BCE as a tie-breaker.

In [ ]:
def train_one(directory, frames, config):
    seed = config['seed']
    seed_everything(seed)
    xval, yval = deterministic_features(frames['val'])
    xtrain = ytrain = None
    if config['mode'] == 'baseline':
        xtrain, ytrain = deterministic_features(frames['train'])
    head = nn.Linear(xval.shape[1], 1).to(DEVICE)
    optimizer = torch.optim.AdamW(head.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()
    state = {'history': [], 'best_auc': -1.0, 'best_loss': 1e30,
             'best_state': None, 'best_epoch': None, 'stale': 0, 'epoch': 0}
    last_path = directory / 'last.pt'
    if last_path.exists():
        if not RESUME:
            raise FileExistsError(f'{last_path} exists; enable RESUME or use another STUDY_NAME.')
        saved = torch.load(last_path, map_location='cpu', weights_only=True)
        if saved['protocol_hash'] != config['protocol_hash']:
            raise ValueError('Checkpoint protocol mismatch.')
        head.load_state_dict(saved['head'])
        optimizer.load_state_dict(saved['optimizer'])
        # AdamW optimizer tensors follow parameters to the active device.
        for opt_state in optimizer.state.values():
            for name, value in opt_state.items():
                if torch.is_tensor(value) and name != 'step':
                    opt_state[name] = value.to(DEVICE)
        state = saved['state']
        print('Resuming after epoch', state['epoch'])
    for epoch in range(state['epoch'] + 1, MAX_EPOCHS + 1):
        if state['stale'] >= PATIENCE:
            break
        seed_everything(seed + epoch * 1009)
        started = time.monotonic()
        if config['mode'] == 'augmented':
            # Fresh stochastic image views; no feature-cache lookup/write here.
            xs, ys = [], []
            loader = image_loader(frames['train'], ImageTransform(PREPROCESS, augmented=True))
            for images, labels in tqdm(loader, desc=f'Augmented features epoch {epoch}', leave=False):
                with torch.no_grad():
                    xs.append(encode_batch(images).cpu())
                ys.append(labels.float())
            xtrain, ytrain = torch.cat(xs), torch.cat(ys)
        head.train()
        train_loader = DataLoader(TensorDataset(xtrain, ytrain), batch_size=HEAD_BATCH_SIZE,
                                  shuffle=True, generator=torch.Generator().manual_seed(seed + epoch))
        total_loss = 0.0
        for features, labels in train_loader:
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(head(features.to(DEVICE)).squeeze(-1), labels.to(DEVICE))
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)
        probabilities = predict_features(head, xval)
        metrics = classification_metrics(yval.numpy(), probabilities)
        loss = metrics['log_loss']
        state['history'].append({'epoch': epoch, 'train_loss': total_loss / len(ytrain),
                                 'seconds': time.monotonic() - started,
                                 **{f'val_{key}': value for key, value in metrics.items()}})
        improved = metrics['auroc'] > state['best_auc'] or (
            metrics['auroc'] == state['best_auc'] and loss < state['best_loss'])
        if improved:
            state.update(best_auc=metrics['auroc'], best_loss=loss, best_epoch=epoch, stale=0,
                         best_state={k: v.detach().cpu().clone() for k, v in head.state_dict().items()})
        else:
            state['stale'] += 1
        state['epoch'] = epoch
        atomic_torch(last_path, {'head': {k: v.detach().cpu() for k,v in head.state_dict().items()},
                                'optimizer': optimizer.state_dict(), 'state': state,
                                'protocol_hash': config['protocol_hash']})
        atomic_csv(directory / 'training_history.csv', pd.DataFrame(state['history']))
        print(f'Epoch {epoch:02d} | train loss {total_loss/len(ytrain):.4f} | '
              f'val AUROC {metrics["auroc"]:.4f} | val F1 {metrics["f1_fake"]:.4f}', flush=True)
    if state['best_state'] is None:
        raise RuntimeError('Training produced no selected checkpoint.')
    head.load_state_dict(state['best_state'])
    atomic_torch(directory / 'best_head.pt', {'head_state_dict': state['best_state'],
                 'feature_dim': int(xval.shape[1]), 'best_epoch': state['best_epoch'],
                 'protocol_hash': config['protocol_hash']})
    export_predictions(directory, 'val', frames['val'], predict_features(head, xval),
                       {'experiment': config['experiment'], 'mode': config['mode'], 'seed': seed})
    history = pd.DataFrame(state['history'])
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
    axes[0].plot(history.epoch, history.train_loss, label='Train')
    axes[0].plot(history.epoch, history.val_log_loss, label='Validation')
    axes[0].set(xlabel='Epoch', ylabel='BCE loss')
    axes[0].legend()
    axes[1].plot(history.epoch, history.val_auroc)
    axes[1].set(xlabel='Epoch', ylabel='Validation AUROC', ylim=(0, 1))
    fig.tight_layout()
    fig.savefig(directory / 'training_curves.png', dpi=150)
    plt.close(fig)
    # Written last: a completed marker means all required outputs exist.
    atomic_json(directory / 'TRAINING_COMPLETE.json', {
        'protocol_hash': config['protocol_hash'], 'best_epoch': state['best_epoch'],
        'completed_utc': utc_now(), 'checkpoint_sha256': file_digest(directory / 'best_head.pt')})

def evaluation_identity(directory, config, condition):
    evaluation_config = {'protocol_hash': config['protocol_hash'],
                         'checkpoint_sha256': file_digest(directory / 'best_head.pt'), 'condition': condition,
                         'degradation': EVAL_DEGRADATIONS if condition != 'clean' else None,
                         'evaluation_seed': EVALUATION_SEED, 'bins': CALIBRATION_BINS,
                         'threshold': THRESHOLD, 'engine_hash': ENGINE_HASH}
    identity = digest_json(evaluation_config)
    return evaluation_config, identity, directory / 'evaluations' / f'{condition}_{identity[:12]}'

def evaluate_one(directory, frames, config, condition):
    best = directory / 'best_head.pt'
    evaluation_config, identity, destination = evaluation_identity(directory, config, condition)
    destination.mkdir(parents=True, exist_ok=True)
    marker = destination / 'COMPLETE.json'
    if marker.exists():
        if json.loads(marker.read_text())['evaluation_hash'] != identity:
            raise ValueError('Evaluation identity collision.')
        for required in ['result.json', 'test_predictions.csv', 'test_metrics.json',
                         'test_calibration.csv', 'test_diagnostics.png', 'evaluation_config.json']:
            if not (destination / required).is_file():
                raise FileNotFoundError(f'Completed evaluation is missing: {destination / required}')
        return
    saved = torch.load(best, map_location='cpu', weights_only=True)
    if saved['protocol_hash'] != config['protocol_hash']:
        raise ValueError('Selected head does not match this run.')
    head = nn.Linear(saved['feature_dim'], 1).to(DEVICE)
    head.load_state_dict(saved['head_state_dict'])
    features, labels = deterministic_features(frames['test'], condition)
    probabilities = predict_features(head, features)
    atomic_json(destination / 'evaluation_config.json', evaluation_config)
    metadata = {'experiment': config['experiment'], 'holdout': config['holdout'],
                'k': config['k'], 'rep': config['rep'], 'seed': config['seed'],
                'mode': config['mode'], 'condition': condition}
    metrics = export_predictions(destination, 'test', frames['test'], probabilities, metadata)
    atomic_json(destination / 'result.json', {**metadata, **metrics,
                 'protocol_hash': config['protocol_hash'], 'evaluation_hash': identity,
                 'family': GENERATOR_FAMILIES.get(config['holdout'], 'unassigned'),
                 'run_dir': str(directory), 'evaluation_dir': str(destination)})
    atomic_json(marker, {'evaluation_hash': identity, 'completed_utc': utc_now()})
    print(f'Test {condition}: accuracy={metrics["accuracy"]:.4f}, '
          f'F1={metrics["f1_fake"]:.4f}, AUROC={metrics["auroc"]:.4f}', flush=True)


## 8. Preview and verify the full experiment plan

This cell validates **all selected manifests before any training**. Missing future
holdouts fail here rather than halfway through a study. Model weights are loaded once.
No output is written into the source dataset. Completed runs from a different study
are never silently imported. The plan records exact manifest checksums and generator sets.

In [ ]:
validate_settings()
ENVIRONMENT = package_versions()
ENGINE_HASH = code_fingerprint()
experiment_names = [f'holdout_{g}_k{k}_rep{r}'
                    for g, k, r in itertools.product(HOLDOUTS, K_VALUES, REPEATS)]
if EXPERIMENT_ALLOWLIST:
    unknown = set(EXPERIMENT_ALLOWLIST) - set(experiment_names)
    if unknown:
        raise ValueError(f'Allowlisted experiments outside selected plan: {unknown}')
    experiment_names = [n for n in experiment_names if n in EXPERIMENT_ALLOWLIST]

FRAMES, EXPERIMENT_INFO = {}, {}
for name in tqdm(experiment_names, desc='Verify manifests/images'):
    FRAMES[name], EXPERIMENT_INFO[name] = load_experiment(name)
ensure_encoder()
PLAN = []
for name, seed, mode in itertools.product(experiment_names, SEEDS, TRAINING_MODES):
    info = EXPERIMENT_INFO[name]
    config = {**info, 'seed': seed, 'mode': mode, 'model': MODEL_NAME,
              'backbone_frozen': True, 'features_l2_normalized': True,
              'label_mapping': {'real': 0, 'fake': 1}, 'threshold': THRESHOLD,
              'augmentation': AUGMENTATION if mode == 'augmented' else None,
              'image_batch_size': IMAGE_BATCH_SIZE, 'head_batch_size': HEAD_BATCH_SIZE,
              'max_epochs': MAX_EPOCHS, 'patience': PATIENCE,
              'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
              'selection': 'validation AUROC; ties use validation BCE',
              'content_hashes_verified': VERIFY_IMAGE_HASHES, 'num_workers': NUM_WORKERS,
              'expected_counts': EXPECTED_COUNTS, 'preprocessing': repr(PREPROCESS),
              'engine_hash': ENGINE_HASH, 'environment': ENVIRONMENT,
              'device_type': DEVICE.type}
    config['protocol_hash'] = digest_json(config)
    directory = OUTPUT_ROOT / 'runs' / f'{name}__{mode}__seed{seed}'
    PLAN.append({'config': config, 'directory': directory})
plan_table = pd.DataFrame([{
    'experiment': p['config']['experiment'], 'mode': p['config']['mode'], 'seed': p['config']['seed'],
    'train_generators': ', '.join(p['config']['train_generators']),
    'training_complete': (p['directory'] / 'TRAINING_COMPLETE.json').exists(),
} for p in PLAN])
display(plan_table)
print(f'{len(PLAN)} selected training runs; output root: {OUTPUT_ROOT}')
print('Test evaluation enabled:', RUN_FINAL_TEST, '| conditions:', EVAL_CONDITIONS)
print('Content hashes verified:', VERIFY_IMAGE_HASHES)


## 9. Run or resume training and final evaluation

This can take time, especially augmented training (new CLIP features each epoch).
Progress and epoch checkpoints are saved throughout. You can interrupt and rerun.
`RUN_TRAINING=False` evaluates existing complete runs only. `RUN_FINAL_TEST=False`
skips test evaluations entirely. Failures appear in `failures/` and the session table.

**Stale lock after a killed job:** inspect `RUNNING.lock`, confirm the previous process/job
has stopped, then remove only that run's lock. Do not remove locks from active runs.
If the protocol differs, use a new `STUDY_NAME`; do not delete completed results to force reuse.

In [ ]:
session_rows = []
active_runs = 0
selected_plan = PLAN
for entry in selected_plan:
    config, directory = entry['config'], entry['directory']
    name = config['experiment']
    directory.mkdir(parents=True, exist_ok=True)
    needs_training = not (directory / 'TRAINING_COMPLETE.json').exists()
    needs_evaluation = RUN_FINAL_TEST and not needs_training and any(
        not (evaluation_identity(directory, config, c)[2] / 'COMPLETE.json').exists()
        for c in EVAL_CONDITIONS)
    needs_work = (RUN_TRAINING and needs_training) or needs_evaluation
    if needs_work and MAX_RUNS_THIS_SESSION is not None and active_runs >= MAX_RUNS_THIS_SESSION:
        session_rows.append({'experiment': name, 'mode': config['mode'], 'seed': config['seed'],
                             'status': 'deferred by session limit'})
        continue
    if needs_work:
        active_runs += 1
    print(f'\n{name} | {config["mode"]} | seed {config["seed"]}', flush=True)
    try:
        with run_lock(directory):
            config_path = directory / 'config.json'
            if config_path.exists():
                previous = json.loads(config_path.read_text())
                if previous['protocol_hash'] != config['protocol_hash']:
                    raise ValueError(f'Protocol changed for {directory}. Choose a new STUDY_NAME.')
            else:
                atomic_json(config_path, {**config, 'created_utc': utc_now(),
                                          'gpu': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None})
            for split in ['train', 'val', 'test']:
                source = MANIFEST_ROOT / name / f'{split}.csv'
                destination = directory / f'input_{split}.csv'
                if file_digest(source) != config['manifest_sha256'][split]:
                    raise ValueError('Input manifest changed since preflight.')
                if destination.exists():
                    if file_digest(destination) != config['manifest_sha256'][split]:
                        raise ValueError(f'Archived manifest changed: {destination}')
                else:
                    destination.write_bytes(source.read_bytes())
            complete = directory / 'TRAINING_COMPLETE.json'
            if complete.exists():
                marker = json.loads(complete.read_text())
                if (marker['protocol_hash'] != config['protocol_hash'] or
                        file_digest(directory / 'best_head.pt') != marker['checkpoint_sha256']):
                    raise ValueError('Completed checkpoint does not match its marker.')
                for required in ['training_history.csv', 'val_predictions.csv', 'val_metrics.json']:
                    if not (directory / required).is_file():
                        raise FileNotFoundError(f'Incomplete completed run: {required}')
                print('Verified completed training; skipping.', flush=True)
            elif RUN_TRAINING:
                train_one(directory, FRAMES[name], config)
            else:
                print('Training incomplete; skipped because RUN_TRAINING=False.')
                session_rows.append({'experiment': name, 'mode': config['mode'], 'seed': config['seed'],
                                     'status': 'training incomplete'})
                continue
            if RUN_FINAL_TEST:
                for condition in EVAL_CONDITIONS:
                    evaluate_one(directory, FRAMES[name], config, condition)
            session_rows.append({'experiment': name, 'mode': config['mode'], 'seed': config['seed'],
                                 'status': 'complete' if RUN_FINAL_TEST else 'trained; test skipped'})
    except Exception as error:
        failure_dir = OUTPUT_ROOT / 'failures'
        failure_dir.mkdir(exist_ok=True)
        atomic_json(failure_dir / f'{uuid.uuid4().hex}.json', {
            'run_dir': str(directory), 'time': utc_now(), 'error': repr(error),
            'traceback': traceback.format_exc()})
        session_rows.append({'experiment': name, 'mode': config['mode'], 'seed': config['seed'],
                             'status': f'failed: {error}'})
        if not CONTINUE_ON_ERROR:
            raise
        print('FAILED:', error, flush=True)
    finally:
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
display(pd.DataFrame(session_rows))


## 10. Collect results, paired augmentation comparisons, and figures

Only the currently selected plan and evaluation settings are collected. Missing results
are listed. Means/std across repeats describe variation in generator choice and sampled
images, not independent confidence intervals. Additional training seeds are separate.
Test images can differ between repeats/k values; augmentation comparisons are paired
within the same experiment and seed. Family summaries average holdout-level results;
they are descriptive and require actual GAN/diffusion holdouts for a family comparison.

In [ ]:
rows, missing = [], []
for entry in PLAN:
    config, directory = entry['config'], entry['directory']
    for condition in EVAL_CONDITIONS:
        candidates = []
        for result_path in (directory / 'evaluations').glob(f'{condition}_*/result.json'):
            destination = result_path.parent
            if not (destination / 'COMPLETE.json').exists():
                continue
            result = json.loads(result_path.read_text())
            evaluation = json.loads((destination / 'evaluation_config.json').read_text())
            if (result['protocol_hash'] == config['protocol_hash']
                    and evaluation['degradation'] == (EVAL_DEGRADATIONS if condition != 'clean' else None)
                    and evaluation['evaluation_seed'] == EVALUATION_SEED
                    and evaluation['bins'] == CALIBRATION_BINS
                    and evaluation['engine_hash'] == ENGINE_HASH):
                candidates.append(result)
        if len(candidates) > 1:
            raise ValueError(f'Ambiguous evaluations for {directory}/{condition}')
        if candidates:
            rows.append(candidates[0])
        else:
            missing.append({'experiment': config['experiment'], 'mode': config['mode'],
                            'seed': config['seed'], 'condition': condition})

SUMMARY_DIR = OUTPUT_ROOT / 'summaries' / digest_json({
    'protocols': [p['config']['protocol_hash'] for p in PLAN], 'conditions': EVAL_CONDITIONS,
    'degradations': EVAL_DEGRADATIONS, 'bins': CALIBRATION_BINS,
    'evaluation_seed': EVALUATION_SEED, 'families': GENERATOR_FAMILIES})[:12]
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
atomic_csv(SUMMARY_DIR / 'missing_results.csv', pd.DataFrame(missing))
print(f'{len(rows)} completed evaluations; {len(missing)} missing. Summary: {SUMMARY_DIR}')
if missing:
    display(pd.DataFrame(missing))
if rows:
    results = pd.DataFrame(rows)
    results['family'] = results.holdout.map(GENERATOR_FAMILIES).fillna('unassigned')
    atomic_csv(SUMMARY_DIR / 'all_results.csv', results)
    metrics = ['accuracy', 'f1_fake', 'auroc', 'recall_fake', 'ece', 'brier', 'log_loss']
    grouped = results.groupby(['holdout', 'k', 'mode', 'condition'], dropna=False)[metrics].agg(['mean', 'std', 'count'])
    grouped.columns = ['_'.join(c) for c in grouped.columns]
    summary = grouped.reset_index()
    atomic_csv(SUMMARY_DIR / 'summary_by_k.csv', summary)
    display(summary)

    # Averaging holdout means first avoids weighting a family by number of repeats.
    family_summary = summary.copy()
    family_summary['family'] = family_summary.holdout.map(GENERATOR_FAMILIES).fillna('unassigned')
    family_summary = family_summary.groupby(['family', 'k', 'mode', 'condition']).agg(
        n_holdouts=('holdout', 'nunique'), mean_auroc=('auroc_mean', 'mean'),
        mean_accuracy=('accuracy_mean', 'mean'), mean_f1=('f1_fake_mean', 'mean')).reset_index()
    atomic_csv(SUMMARY_DIR / 'family_summary.csv', family_summary)

    keys = ['experiment', 'holdout', 'k', 'rep', 'seed', 'condition']
    base = results[results['mode'] == 'baseline'][keys + metrics]
    aug = results[results['mode'] == 'augmented'][keys + metrics]
    paired = base.merge(aug, on=keys, suffixes=('_baseline', '_augmented'), validate='one_to_one')
    for metric in metrics:
        paired[metric + '_delta'] = paired[metric + '_augmented'] - paired[metric + '_baseline']
    atomic_csv(SUMMARY_DIR / 'augmentation_deltas.csv', paired)

    for (holdout, condition), table in summary.groupby(['holdout', 'condition']):
        fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
        for ax, metric in zip(axes, ['accuracy', 'f1_fake', 'auroc']):
            for mode, subset in table.groupby('mode'):
                subset = subset.sort_values('k')
                ax.errorbar(subset.k, subset[metric+'_mean'], yerr=subset[metric+'_std'].fillna(0),
                            marker='o', capsize=3, label=mode)
            ax.set(xlabel='Number of training generators (k)', ylabel=metric, xticks=K_VALUES, ylim=(0,1))
            ax.grid(alpha=0.2)
        axes[0].legend()
        fig.suptitle(f'Held out: {holdout} | {condition} | mean +/- sample SD across available repeats/seeds')
        fig.tight_layout()
        fig.savefig(SUMMARY_DIR / f'{holdout}_{condition}_comparison.png', dpi=150)
        plt.show()

    # Heatmap is useful with one holdout now and expands as more holdouts are added.
    for (mode, condition), table in summary.groupby(['mode', 'condition']):
        pivot = table.pivot(index='holdout', columns='k', values='auroc_mean')
        fig, ax = plt.subplots(figsize=(6, max(2.2, len(pivot)*0.6)))
        values = pivot.to_numpy()
        plotted = ax.imshow(values, vmin=0, vmax=1, cmap='Blues', aspect='auto')
        ax.set(xticks=range(len(pivot.columns)), xticklabels=pivot.columns,
               yticks=range(len(pivot.index)), yticklabels=pivot.index,
               xlabel='Training generators (k)', ylabel='Held-out generator',
               title=f'{mode} | {condition} | mean test AUROC')
        for (i,j), value in np.ndenumerate(values):
            if np.isfinite(value):
                ax.text(j,i,f'{value:.3f}',ha='center',va='center',color='white' if value>0.6 else 'black')
        fig.colorbar(plotted, ax=ax)
        fig.tight_layout()
        fig.savefig(SUMMARY_DIR / f'{mode}_{condition}_auroc_heatmap.png', dpi=150)
        plt.show()
else:
    print('No matching final test results yet. Training/validation artifacts remain in each run folder.')


## 11. What to share with the CNN team

- This notebook and the **same exact experiment CSVs** (compare archived manifest hashes).
- Settings, `config.json`, `training_history.csv`, `best_head.pt`, and `last.pt` on CARC.
- Validation/test prediction CSVs, metrics, reliability diagrams, and summary CSVs/figures.
- CNN exports should use `image_id` (SHA-256), `label`, `source_generator`, `prob_fake`,
  `experiment`, `seed`, `mode`, and `condition`, with fake=1 and threshold=0.5.
- Coordinate the augmentation recipe, its position after resize/crop, and degradation
  seeds. Preserve each pretrained encoder's required normalization. Keep training budgets
  fixed across k. Do not use reported test results to retune this protocol.
- `baseline` versus `augmented` is **one combined augmentation comparison**. Individual
  augmentation ablations require additional separately named training modes/studies.
- Current default is ADM only. Extra holdouts require the corresponding CSV folders;
  no images or manifests are generated by this notebook.
- Frozen CLIP versus fine-tuned ResNet compares complete detection approaches; it does
  not isolate architecture alone. Domain-adversarial/contrastive training and zero-shot
  LLM/deepfake evaluation are separate experiments, not implemented here.

Large checkpoints/features stay on CARC. Commit this notebook and small summary files
to GitHub. No dataset files are modified.

References: [OpenAI CLIP](https://github.com/openai/CLIP),
[UniversalFakeDetect paper](https://openaccess.thecvf.com/content/CVPR2023/html/Ojha_Towards_Universal_Fake_Image_Detectors_That_Generalize_Across_Generative_Models_CVPR_2023_paper.html).